# Проверка исходных данных

Цель - убедиться, что файл можно использовать для расчёта витрин.

Проверяем размер, схему, пропуски, типы событий, время и полные повторы строк.

## Правила расчёта

- `timestamp` — глобальное время в секундах с точностью до 5 секунд.
- Условный день: `timestamp // 86400`.
- Listen+: прослушано больше 50% трека.
- Повтор: `played_ratio_pct > 100`.
- Новая сессия начинается после перерыва больше 30 минут.

In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE = PROJECT_ROOT / "data" / "yambda" / "flat" / "50m" / "multi_event.parquet"
MARTS = PROJECT_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 30)

from src.prepare_events import prepare_events

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
assert SOURCE.exists(), "Сначала запустите: python scripts/download_data.py"
con = duckdb.connect()

## Схема

In [2]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{SOURCE.as_posix()}')").df()
schema[["column_name", "column_type", "null"]].rename(columns={
    "column_name": "столбец", "column_type": "тип", "null": "может быть пустым"
})

,столбец,тип,может быть пустым
0,uid,UINTEGER,YES
1,timestamp,UINTEGER,YES
2,item_id,UINTEGER,YES
3,is_organic,UTINYINT,YES
4,played_ratio_pct,USMALLINT,YES
5,track_length_seconds,UINTEGER,YES
6,event_type,VARCHAR,YES


## Основные проверки

In [3]:
quality = con.execute(f"""
SELECT
    count(*) AS events,
    count(DISTINCT uid) AS users,
    count(DISTINCT item_id) AS tracks,
    min(timestamp) AS min_timestamp,
    max(timestamp) AS max_timestamp,
    count(*) FILTER (WHERE uid IS NULL OR item_id IS NULL OR timestamp IS NULL
                     OR is_organic IS NULL OR event_type IS NULL) AS missing_required,
    count(*) FILTER (WHERE timestamp % 5 <> 0) AS invalid_timestamp_step,
    count(*) FILTER (WHERE event_type = 'listen' AND
                     (played_ratio_pct IS NULL OR track_length_seconds IS NULL)) AS invalid_listens,
    count(*) FILTER (WHERE event_type <> 'listen' AND
                     (played_ratio_pct IS NOT NULL OR track_length_seconds IS NOT NULL)) AS invalid_actions
FROM read_parquet('{SOURCE.as_posix()}')
""").df()

quality.rename(columns={
    "events": "события", "users": "пользователи", "tracks": "треки",
    "min_timestamp": "минимальное время", "max_timestamp": "максимальное время",
    "missing_required": "пропуски в обязательных полях",
    "invalid_timestamp_step": "ошибки шага времени",
    "invalid_listens": "ошибочные прослушивания",
    "invalid_actions": "ошибочные реакции",
})

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,события,пользователи,треки,минимальное время,максимальное время,пропуски в обязательных полях,ошибки шага времени,ошибочные прослушивания,ошибочные реакции
0,47790449,10000,934057,0,26000000,0,0,0,0


## События и повторы

In [4]:
events = con.execute(f"""
SELECT event_type AS event, count(*) AS rows
FROM read_parquet('{SOURCE.as_posix()}')
GROUP BY event_type
ORDER BY rows DESC
""").df()

duplicates = con.execute(f"""
SELECT coalesce(sum(rows - 1), 0) AS duplicate_rows
FROM (
    SELECT count(*) AS rows
    FROM read_parquet('{SOURCE.as_posix()}')
    GROUP BY uid, item_id, timestamp, is_organic, event_type,
             played_ratio_pct, track_length_seconds
    HAVING count(*) > 1
)
""").fetchone()[0]

display(events.rename(columns={"event": "тип события", "rows": "строки"}))
print(f"Полных повторов строк: {duplicates:,}".replace(",", " "))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,тип события,строки
0,listen,46467212
1,like,881456
2,unlike,312972
3,dislike,107776
4,undislike,21033


Полных повторов строк: 236 230


## Подготовка общей таблицы

Код подготовки находится в `src/prepare_events.py`. Здесь рассчитываются признаки,
которые используются при построении витрин: день, Listen+, повтор, источник,
время воспроизведения и начало сессии.

In [5]:
source_report = prepare_events(SOURCE, STAGE_DB)
stage = duckdb.connect(str(STAGE_DB), read_only=True)
stage.execute("DESCRIBE stage_events").df()[["column_name", "column_type"]].rename(
    columns={"column_name": "столбец", "column_type": "тип"}
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,столбец,тип
0,uid,UINTEGER
1,item_id,UINTEGER
2,timestamp,UINTEGER
3,day_idx,UINTEGER
4,is_organic,UTINYINT
5,event_type,VARCHAR
6,played_ratio_pct,USMALLINT
7,track_length_seconds,UINTEGER
8,is_listen,BOOLEAN
9,is_listen_plus,BOOLEAN


## Вывод

- В файле 47 790 449 событий, 10 000 пользователей и 934 057 треков.
- Обязательные поля заполнены, значения времени корректны.
- 236 230 полных повторов не удаляем автоматически: одинаковые события могут быть реальными повторными действиями.
- Данные подходят для первого этапа анализа.